In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import yaml

# import orjson
import re
import ast

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    'font.serif': ['Computer Modern'],
})

import re

def tex_safe(text):
    # This keeps your source 'text' original, but returns a
    # version LaTeX won't choke on.
    return text.replace("%", r"\%")#.replace("_", r"\_")

# --- Point at the project-local TeX Live install ---
import glob
import shutil
import subprocess

TEXDIR = "/data/snoplus/weiiiiiii/aiproj/driftmtplm/texlive"
tex_bin_candidates = glob.glob(os.path.join(TEXDIR, "bin", "*"))
assert tex_bin_candidates, f"No TeX Live bin/<platform> directory found under {TEXDIR}"
tex_bin = tex_bin_candidates[0]
if tex_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = tex_bin + os.pathsep + os.environ["PATH"]

# --- Verify the toolchain resolves from this TeX Live install ---
for exe in ("latex", "pdflatex", "dvipng"):
    resolved = shutil.which(exe)
    assert resolved, f"{exe} not found on PATH -- check {tex_bin}"
    flag = "" if resolved.startswith(TEXDIR) else "  [WARNING: resolved outside the project TeX Live install]"
    print(f"{exe:8s} -> {resolved}{flag}")

# --- Confirm the specific tlmgr packages the figures rely on are actually installed ---
required_packages = [
    "cm-super", "type1cm", "dvipng", "amsmath", "amsfonts",
    "symbol", "listings", "xcolor", "underscore", "geometry", "graphics",
]
tlmgr = os.path.join(tex_bin, "tlmgr")
missing = [
    name for name in required_packages
    if subprocess.run(
        [tlmgr, "info", "--only-installed", "--data", "name", name],
        capture_output=True, text=True,
    ).stdout.strip() == ""
]
if missing:
    raise RuntimeError(f"Missing LaTeX packages: {missing}. Install with: {tlmgr} install {' '.join(missing)}")
print("All required TeX Live packages are installed.")

# --- End-to-end smoke test: actually render text through the usetex pipeline ---
_fig, _ax = plt.subplots()
_ax.set_title(r"\textbf{Smoke test: } $\alpha^2 + \beta_i$ \% \_ \#")
_fig.savefig(os.path.join(os.path.expanduser("~"), ".cache_usetex_smoketest.pdf"))
plt.close(_fig)
print("matplotlib text.usetex rendering succeeded.")


latex    -> /data/snoplus/weiiiiiii/aiproj/driftmtplm/texlive/bin/x86_64-linux/latex
pdflatex -> /data/snoplus/weiiiiiii/aiproj/driftmtplm/texlive/bin/x86_64-linux/pdflatex
dvipng   -> /data/snoplus/weiiiiiii/aiproj/driftmtplm/texlive/bin/x86_64-linux/dvipng
All required TeX Live packages are installed.
matplotlib text.usetex rendering succeeded.


In [9]:
BASE_RAW_DATA_DIR = "/home/huangp/aiproj/driftmtplm/third_party/mtp-lm/figures_data_raw"
BASE_FIGURE_DIR = "/home/huangp/aiproj/driftmtplm/third_party/mtp-lm/auto_figures"

In [12]:
RAW_DATA_FILENAME = "/home/huangp/aiproj/driftmtplm/third_party/mtp-lm/figures_data_raw/singleshot-evals_summary_table_20260821-115450.csv"

# Data prep

In [13]:
raw_df = pd.read_csv(os.path.join(BASE_RAW_DATA_DIR, RAW_DATA_FILENAME), index_col=0)
raw_df.head()

,summary,config,name,tags
0,"{'_step': 60000, '_runtime': 1.992514575, 'run...","{'date': 1787086633.40845, 'config': {'limit':...",l3_magpie_metamath,"['daint', 'manual_pusher', 'stepwise']"


In [ ]:
flat_df = raw_df.copy()

wandb_pull_cols = ["summary", "config", "name", "tags"]

# This regex only hits 'true', 'false', 'null' when they are NOT part of another word
def clean_and_eval(s):
    if not isinstance(s, str): return s
    # \b ensures we don't hit "true_val" or "is_true"
    s = re.sub(r'\btrue\b', 'True', s)
    s = re.sub(r'\bfalse\b', 'False', s)
    s = re.sub(r'\bnull\b', 'None', s)
    try:
        return ast.literal_eval(s)
    except:
        return s

# The One-Liner for multiple columns
flat_df[wandb_pull_cols] = flat_df[wandb_pull_cols].map(clean_and_eval)
print(flat_df[wandb_pull_cols].head())

def flatten_column(df, col_name, sep="_"):
    # Normalize the dicts into a dataframe
    flat_col = pd.json_normalize(df[col_name].tolist(), sep=sep)
    
    # Optional: Prefix new columns with the original column name to avoid collisions
    flat_col.columns = [f"{col_name}{sep}{c}" for c in flat_col.columns]
    
    # Drop the original and join the new columns
    return df.drop(columns=[col_name]).join(flat_col.reset_index(drop=True))

# only summary/config are actually dict-shaped (pulled straight from the wandb
# API); name is a plain string and tags is a list, so json_normalize chokes on
# them -- they get carried through untouched below instead.
dict_cols = ["summary", "config"]
for col in dict_cols:
    flat_df = flatten_column(flat_df, col)

flat_df["name"] = raw_df["name"]
flat_df["tags"] = raw_df["tags"]

In [ ]:
# shape of df
print("Raw DF shape:", raw_df.shape)
print("Flat DF shape:", flat_df.shape)
flat_df.tail()

In [ ]:
# # just poking for columns
# # keystr = "rougeL"
# # keystr = "ifeval"
# # keystr = "tps"
# keystr = "mtp_toks_gend_incl_prefillplus1"
# found_cols = [c for c in flat_df.columns.tolist() if keystr in c]
# found_cols

# Tables

In [16]:
# 1. Task Configuration
TASK_CONFIGS = {
    "gsm8k_cot_singleshot": {
        "pretty_name": "GSM8K",# (CoT,N-shot)",
        "perf_style": "acc_pct",
        "perf_col": "summary_gsm8k_cot_singleshot/exact_match,flexible-extract",
        "perf_std_col": "summary_gsm8k_cot_singleshot/exact_match_stderr,flexible-extract",
        "eff_col": "summary_gsm8k_cot_singleshot/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_gsm8k_cot_singleshot/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_gsm8k_cot_singleshot/samples/mtp_tps/mean",
        "tps_std_col": "summary_gsm8k_cot_singleshot/samples/mtp_tps/std",
        "toks_gend_col": "summary_gsm8k_cot_singleshot/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_gsm8k_cot_singleshot/samples/mtp_toks_gend_incl_prefillplus1/std",
    },
    "aime25": {
        "pretty_name": "AIME25",
        "perf_style": "acc_pct",
        "perf_col": "summary_aime25/exact_match,none",
        "perf_std_col": "summary_aime25/exact_match_stderr,none",
        "eff_col": "summary_aime25/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_aime25/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_aime25/samples/mtp_tps/mean",
        "tps_std_col": "summary_aime25/samples/mtp_tps/std",
        "toks_gend_col": "summary_aime25/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_aime25/samples/mtp_toks_gend_incl_prefillplus1/std",
    },
    "bbh_cot_fewshot": {
        "pretty_name": "BBH", # (N-shot)",
        "perf_style": "acc_pct",
        "perf_col": "summary_bbh_cot_fewshot/exact_match,get-answer",
        "perf_std_col": "summary_bbh_cot_fewshot/exact_match_stderr,get-answer",
        "eff_col": "summary_bbh_cot_fewshot/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_bbh_cot_fewshot/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_bbh_cot_fewshot/samples/mtp_tps/mean",
        "tps_std_col": "summary_bbh_cot_fewshot/samples/mtp_tps/std",
        "toks_gend_col": "summary_bbh_cot_fewshot/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_bbh_cot_fewshot/samples/mtp_toks_gend_incl_prefillplus1/std",
    },
    "ifeval": {
        "pretty_name": "IFEval", # (Prompt-lvl,loose)",
        "perf_style": "acc_pct",
        "perf_col": "summary_ifeval/prompt_level_loose_acc,none",
        "perf_std_col": "summary_ifeval/prompt_level_loose_acc_stderr,none",
        "eff_col": "summary_ifeval/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_ifeval/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_ifeval/samples/mtp_tps/mean",
        "tps_std_col": "summary_ifeval/samples/mtp_tps/std",
        "toks_gend_col": "summary_ifeval/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_ifeval/samples/mtp_toks_gend_incl_prefillplus1/std",
    },
    "cnn_dailymail_abisee": {
        "pretty_name": "CNN DailyMail",# (ROUGE-L)",
        "perf_style": "rouge",
        "perf_col": "summary_cnn_dailymail_abisee/rougeL,none",
        "perf_std_col": "summary_cnn_dailymail_abisee/rougeL_stderr,none",
        "eff_col": "summary_cnn_dailymail_abisee/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_cnn_dailymail_abisee/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_cnn_dailymail_abisee/samples/mtp_tps/mean",
        "tps_std_col": "summary_cnn_dailymail_abisee/samples/mtp_tps/std",
        "toks_gend_col": "summary_cnn_dailymail_abisee/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_cnn_dailymail_abisee/samples/mtp_toks_gend_incl_prefillplus1/std",
    },
    "gpqa_main_cot_n_shot": {
        "pretty_name": "GPQA Main", # (CoT,N-shot)",
        "perf_style": "acc_pct",
        "perf_col": "summary_gpqa_main_cot_n_shot/exact_match,flexible-extract",
        "perf_std_col": "summary_gpqa_main_cot_n_shot/exact_match_stderr,flexible-extract",
        "eff_col": "summary_gpqa_main_cot_n_shot/samples/mtp_avg_effective_k_trimmed/mean",
        "eff_std_col": "summary_gpqa_main_cot_n_shot/samples/mtp_avg_effective_k_trimmed/std",
        "tps_col": "summary_gpqa_main_cot_n_shot/samples/mtp_tps/mean",
        "tps_std_col": "summary_gpqa_main_cot_n_shot/samples/mtp_tps/std",
        "toks_gend_col": "summary_gpqa_main_cot_n_shot/samples/mtp_toks_gend_incl_prefillplus1/mean",
        "toks_gend_std_col": "summary_gpqa_main_cot_n_shot/samples/mtp_toks_gend_incl_prefillplus1/std",
    }
}

TASK_PERF_MAP = {
    "acc_pct": {
        "pretty_name": "Acc. (%)",
        "round":1,
    },
    "rouge": {
        "pretty_name": "ROUGE-L",
        "round":3,
    }
}

# 2. Filter Naming Configuration
FILTER_PRETTY_NAMES = {
    "config_chat_template": "Chat Template",
    "config_config_gen_kwargs_max_length": "Max Length"
}

# 3. Strategy Pretty Mapping
STRATEGY_MAP = {
    "nan": "Static",
    **{f"['conf_adapt', {t}]": f"ConfAdapt ($\\tau={t}$)"
         for t in ["0.995", "0.99", "0.98", "0.97", "0.96", "0.95", "0.9", "0.87", "0.85", "0.8", "0.75", "0.7", "0.65", "0.6"]}
}

# 4. Strategy Order (Raw identifiers used for logic, pretty names applied at end)
STRATEGY_ORDER = [
    "nan k=1",
    "nan k=2",
    "nan k=3",
    "nan k=4",
    "nan k=5",
    # "nan k=8",
    # "nan k=16",
    "['conf_adapt', 0.995] k=16",
    "['conf_adapt', 0.99] k=16",
    "['conf_adapt', 0.98] k=16",
    "['conf_adapt', 0.97] k=16",
    "['conf_adapt', 0.96] k=16",
    "['conf_adapt', 0.95] k=16",
    "['conf_adapt', 0.9] k=16",
    "['conf_adapt', 0.87] k=16",
    "['conf_adapt', 0.85] k=16",
    "['conf_adapt', 0.8] k=16",
    "['conf_adapt', 0.75] k=16",
    "['conf_adapt', 0.7] k=16",
    "['conf_adapt', 0.65] k=16",
    "['conf_adapt', 0.6] k=16",
]

# 5. Model Configuration
MODEL_CONFIGS = [
    {
        "id": "daint_prod_ift_mask_fix_1N4n_9d30cad5",
        "pretty_name": "L3.1-8B-Magpie",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_ift_q3-4b_1N4n_16cdce0f",
        "pretty_name": "Qwen3-4B-Inst-2507",
        "filters": {
            "config_chat_template": "is_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    # {
    #     "id": "daint_prod_ift_q3-4b_1N4n_16cdce0f",
    #     "pretty_name": "Qwen3-4B-Inst-2507",
    #     "filters": {
    #         "config_chat_template": "is_null",
    #         "config_config_gen_kwargs_max_length": 2048
    #     }
    # },
    {
        "id": "daint_prod_ift_magpie_1N4n_44004b35",
        "pretty_name": "L3.1-8B-Magpie FT Magpie",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_32112117",
        "pretty_name": "L3.1-8B-Magpie Beta 2.0.",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_82938517",
        "pretty_name": "L3.1-8B-Magpie GT Suprv.",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_b258ae13",
        "pretty_name": "L3.1-8B-Magpie BDA",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_d2e5a6cb",
        "pretty_name": "L3.1-8B-Magpie Beta 1.0",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_d30c404e",
        "pretty_name": "L3.1-8B-Magpie Soft Teacher",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_e12fd460",
        "pretty_name": "L3.1-8B-Magpie Static k",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_suprv_abl_1N4n_fcdeefba",
        "pretty_name": "L3.1-8B-Magpie Prefix Loss",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    {
        "id": "daint_prod_pre_arxiv_extra_1N4n_8aa66673",
        "pretty_name": "L3.1-8B-Magpie Static k=9",
        "filters": {
            "config_chat_template": "is_not_null",
            "config_config_gen_kwargs_max_length": "is_null"
        }
    },
    
]

def format_filter_string(filter_dict):
    parts = []
    for col, val in filter_dict.items():
        name = FILTER_PRETTY_NAMES.get(col)
        if col == "config_chat_template":
            status = "On" if val == "is_not_null" else "Off"
            parts.append(f"{name}: {status}")
        elif col == "config_config_gen_kwargs_max_length" and val != "is_null":
            parts.append(f"{name}: {val}")
    # Using a standard newline character for logic; we'll convert to LaTeX later
    return "\\n" + "\\n".join(parts) if parts else ""

def apply_custom_filters(df, filter_dict):
    filtered_df = df.copy()
    for col, value in (filter_dict or {}).items():
        if col not in filtered_df.columns: continue
        if value == "is_null": filtered_df = filtered_df[filtered_df[col].isnull()]
        elif value == "is_not_null": filtered_df = filtered_df[~filtered_df[col].isnull()]
        else: filtered_df = filtered_df[filtered_df[col] == value]
    return filtered_df

def create_hierarchical_summary_table(
    df, 
    tasks, 
    model_configs,
    sort_by="custom", 
    sort_task=None,   
    step_col="summary__step",
    use_latest_step=True,
    include_step=False,
    include_tps=False,
    include_tot_toks=False,
    show_baseline=True, # New flag for upperbound baseline
    baseline_str="Baseline Step 0, k=1",
    rounding_cfg={"Eff. k": 1, "Toks/s": 1}
):
    results = []
    strategy_key = "config_config_gen_kwargs_strategy"
    k_key = "config_config_gen_kwargs_k_toks"
    ordered_model_names = []
    ordered_task_names = [TASK_CONFIGS[t]["pretty_name"] for t in tasks if t in TASK_CONFIGS]

    for model_cfg in model_configs:
        model_id = model_cfg["id"]
        filter_suffix = format_filter_string(model_cfg.get("filters", {}))
        full_row_name = f"{model_cfg['pretty_name']}{filter_suffix}".strip()
        ordered_model_names.append(full_row_name)
        
        for task_key in tasks:
            t_cfg = TASK_CONFIGS[task_key]
            subset = df[(df["name"] == model_id) & (~df[t_cfg["perf_col"]].isnull())].copy()
            subset = apply_custom_filters(subset, model_cfg.get("filters", {}))
            
            if subset.empty: continue
            subset[strategy_key] = subset[strategy_key].astype(str)
            subset[k_key] = subset[k_key].astype(int)

            # --- BASELINE LOGIC ---
            if show_baseline:
                # Baseline is usually defined as k=1 at the very first step (0)
                baseline_group = subset[(subset[k_key] == 1) & (subset[step_col] == 0)]
                if not baseline_group.empty:
                    row_data = baseline_group.iloc[0]
                    t_perf_sty = t_cfg["perf_style"]
                    perf_round = TASK_PERF_MAP[t_perf_sty]["round"]
                    perf_pretty = TASK_PERF_MAP[t_perf_sty]["pretty_name"]
                    
                    perf_value = np.round(row_data.get(t_cfg["perf_col"], 0) * (100 if "%" in perf_pretty else 1), perf_round)
                    perf_err_value = row_data.get(t_cfg["perf_std_col"], np.nan)
                    if not np.isnan(perf_err_value):
                        perf_err_value = np.round(perf_err_value * (100 if "%" in perf_pretty else 1), perf_round)
                        perf_value = f"{perf_value} ± {perf_err_value}"

                    task_tps_col = t_cfg["tps_col"]
                    tps_value = np.round(row_data.get(task_tps_col, np.nan), rounding_cfg.get("Toks/s", 1))
                    tps_err_value = row_data.get(t_cfg["tps_std_col"], np.nan)
                    if not np.isnan(tps_err_value):
                        tps_err_value = np.round(tps_err_value, rounding_cfg.get("Toks/s", 1))
                        tps_value = f"{tps_value} ± {tps_err_value}"
                    
                    toks_gend_col = t_cfg["toks_gend_col"]
                    toks_gend_value = np.round(row_data.get(toks_gend_col, np.nan), rounding_cfg.get("Total Toks", 0))
                    toks_gend_err_value = row_data.get(t_cfg["toks_gend_std_col"], np.nan)
                    if not np.isnan(toks_gend_err_value):
                        toks_gend_err_value = np.round(toks_gend_err_value, rounding_cfg.get("Total Toks", 0))
                        toks_gend_value = f"{toks_gend_value} ± {toks_gend_err_value}"
                    
                    results.append({
                        "Model": full_row_name,
                        "Strategy": baseline_str,
                        "LogicID": "baseline",
                        "Step": 0,
                        "Task": t_cfg["pretty_name"],
                        "Perf_Internal": perf_value,
                        "Eff. k": 1,
                        "Toks/s": tps_value,
                        "Tot. Toks": toks_gend_value,
                    })
            # ----------------------
            
            for (strat, k_val), group in subset.groupby([strategy_key, k_key]):
                # Skip if this is the k=1 row and we already handled it as baseline (optional, keeps strategy list clean)
                if show_baseline and k_val == 1 and group[step_col].min() == 0 and use_latest_step:
                    # If you want to show the k=1 latest step alongside baseline, don't skip
                    pass

                strat_pretty = STRATEGY_MAP.get(strat, strat)
                strat_label = f"{strat_pretty} k={k_val}" # if strat != "nan" else f"k={k_val}"
                strat_logic_id = f"{strat} k={k_val}"# if strat != "nan" else f"k={k_val}"
                
                if sort_by == "custom" and strat_logic_id not in STRATEGY_ORDER: continue

                target_step = group[step_col].max() if use_latest_step else group[step_col].iloc[-1]
                row_data = group[group[step_col] == target_step].iloc[0]

                t_perf_sty = t_cfg["perf_style"]
                perf_round = TASK_PERF_MAP[t_perf_sty]["round"]
                perf_pretty = TASK_PERF_MAP[t_perf_sty]["pretty_name"]

                perf_value = np.round(row_data.get(t_cfg["perf_col"], 0) * (100 if "%" in perf_pretty else 1), perf_round)
                perf_err_value = row_data.get(t_cfg["perf_std_col"], np.nan)
                if not np.isnan(perf_err_value):
                    perf_err_value = np.round(perf_err_value * (100 if "%" in perf_pretty else 1), perf_round)
                    perf_value = f"{perf_value} ± {perf_err_value}"

                # task_eff_col = t_cfg["eff_col"]
                # if any(key in model_id for key in ["9d30cad5","16cdce0f","44004b35"]):
                #     task_eff_col = task_eff_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")
                task_eff_col = t_cfg["eff_col"]
                task_eff_err_col = t_cfg["eff_std_col"]
                if row_data[task_eff_col] is pd.NA or row_data[task_eff_col] == np.nan or str(row_data[task_eff_col]) == "nan":
                    task_eff_col = task_eff_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")
                    task_eff_err_col = task_eff_err_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")
                
                eff_value = np.round(row_data.get(task_eff_col, np.nan), rounding_cfg.get("Eff. k", 1))
                eff_err_value = row_data.get(task_eff_err_col, np.nan)
                # only do for adaptive schemes
                if not np.isnan(eff_err_value) and strat != "nan":
                    eff_err_value = np.round(eff_err_value, rounding_cfg.get("Eff. k", 1))
                    eff_value = f"{eff_value} ± {eff_err_value}"
                if strat == "nan":
                    eff_value = f"{int(eff_value)}"


                task_tps_col = t_cfg["tps_col"]
                tps_value = np.round(row_data.get(task_tps_col, np.nan), rounding_cfg.get("Toks/s", 1))
                tps_err_value = row_data.get(t_cfg["tps_std_col"], np.nan)
                if not np.isnan(tps_err_value):
                    tps_err_value = np.round(tps_err_value, rounding_cfg.get("Toks/s", 1))
                    tps_value = f"{tps_value} ± {tps_err_value}"
                
                toks_gend_col = t_cfg["toks_gend_col"]
                toks_gend_value = np.round(row_data.get(toks_gend_col, np.nan), rounding_cfg.get("Total Toks", 0))
                toks_gend_err_value = row_data.get(t_cfg["toks_gend_std_col"], np.nan)
                if not np.isnan(toks_gend_err_value):
                    toks_gend_err_value = np.round(toks_gend_err_value, rounding_cfg.get("Total Toks", 0))
                    toks_gend_value = f"{toks_gend_value} ± {toks_gend_err_value}"
                
                results.append({
                    "Model": full_row_name,
                    "Strategy": strat_label,
                    "LogicID": strat_logic_id,
                    "Step": int(target_step),
                    "Task": t_cfg["pretty_name"],
                    "Perf_Internal": perf_value,
                    "Eff. k": eff_value,
                    "Toks/s": tps_value,
                    "Tot. Toks": toks_gend_value,
                })
            
    res_df = pd.DataFrame(results)
    pivot_vals = ["Perf_Internal", "Eff. k"] + (["Step"] if include_step else []) + (["Toks/s"] if include_tps else []) + (["Tot. Toks"] if include_tot_toks else [])
    table = res_df.pivot(index=["Model", "Strategy", "LogicID"], columns="Task", values=pivot_vals)
    table = table.reorder_levels([1, 0], axis=1).reindex(columns=ordered_task_names, level=0)

    # Rename Perf_Internal to task-specific names
    new_cols = []
    for task_pretty, metric in table.columns:
        if metric == "Perf_Internal":
            task_key = next(k for k, v in TASK_CONFIGS.items() if v["pretty_name"] == task_pretty)
            new_cols.append((task_pretty, TASK_PERF_MAP[TASK_CONFIGS[task_key]["perf_style"]]["pretty_name"]))
        else:
            new_cols.append((task_pretty, metric))
    table.columns = pd.MultiIndex.from_tuples(new_cols)

    if sort_by == "custom":
        pretty_order = []
        if show_baseline: pretty_order.append(baseline_str)
        
        for s_id in STRATEGY_ORDER:
            if " k=" in s_id:
                s_part, k_part = s_id.split(" k=")
                pretty_s = STRATEGY_MAP.get(s_part, s_part)
                pretty_order.append(f"{pretty_s} k={k_part}")# if s_part != "nan" else f"k={k_part}")
            else:
                pretty_order.append(s_id)
        
        new_index = pd.MultiIndex.from_product([ordered_model_names, pretty_order], names=["Model", "Strategy"])
        table = table.reset_index(level="LogicID", drop=True)
        table = table.reindex(new_index).dropna(how='all')

        # now strip the k= part from all the strategies that are ConfAdapt for cleaner display
        def clean_strategy_name(s):
            if "ConfAdapt" in s and " k=" in s:
                return s.split(" k=")[0]
            return s
        table = table.rename(index=lambda x: clean_strategy_name(x) if isinstance(x, str) else x, level="Strategy") 
    else:
        # Dynamic metric sorting (keep baseline at top)
        table = table.reset_index().drop(columns=['LogicID'])
        model_sorter = {name: i for i, name in enumerate(ordered_model_names)}
        table['model_rank'] = table['Model'].map(model_sorter)
        table['is_baseline'] = table['Strategy'] == baseline_str
        
        target_pretty_task = TASK_CONFIGS[sort_task or tasks[0]]["pretty_name"]
        perf_name = TASK_PERF_MAP[TASK_CONFIGS[sort_task or tasks[0]]["perf_style"]]["pretty_name"]
        metric = perf_name if sort_by == "perf" else "Eff. k"
        
        sort_col = (target_pretty_task, metric)
        # Sort by Model Rank, then Baseline status, then metric
        table = table.sort_values(['model_rank', 'is_baseline', sort_col], 
                                 ascending=[True, False, False if sort_by=="perf" else True])
        table = table.drop(columns=['model_rank', 'is_baseline']).set_index(['Model', 'Strategy'])

    return table

def dump_to_latex(summary_table, longtable=False):

    if summary_table.empty:
        print("Summary table is empty. No LaTeX to generate.")
        return
    # Ensure index names are empty to avoid alignment shifts
    summary_table.index.names = [None, None]

    # make all data strings for better LaTeX control
    summary_table = summary_table.astype(str)
    
    # 1. Generate initial LaTeX
    latex_code = summary_table.to_latex(
        index=True, 
        multirow=True, 
        multicolumn=True, 
        # float_format=lambda x: f"{x:g}",
        caption="Model evaluation summary with row headers split across rows.",
        label=latex_table_name,
        escape=False,
        column_format="ll"+"c" * (len(summary_table.columns)),
    )
    
    # 2. Cleanup basic LaTeX breaks
    latex_code = latex_code.replace("%", "\%")
    latex_code = latex_code.replace("_", "\_")
    
    # --- FIXED LINE-BY-LINE ALIGNMENT LOGIC ---
    def distribute_multirow_lines(latex_str):
        # Split into lines to process row by row
        lines = latex_str.split('\n')
        final_lines = []
        
        i = 0
        while i < len(lines):
            line = lines[i]
            # Detect multirow with our newline marker
            if '\\multirow' in line and '\\n' in line:
                # Find the multirow command and the number of rows it spans
                # format: \multirow[t]{8}{*}{Name\n(Filter)}
                match = re.search(r'\\multirow\[t\]\{(\d+)\}\{\*\}\{(.*?)\}', line)
                if match:
                    total_rows_spanned = int(match.group(1))
                    raw_content = match.group(2)
                    
                    # Split the model header by our newline marker
                    parts = [p.strip() for p in raw_content.split('\\n')]
                    
                    # Replace the multirow command in the current line with just the first part
                    line = line.replace(match.group(0), parts[0])
                    final_lines.append(line)
                    
                    # Now inject the remaining parts into subsequent rows
                    for p_idx in range(1, len(parts)):
                        next_row_idx = i + p_idx
                        if next_row_idx < len(lines):
                            # In Pandas LaTeX, subsequent rows in a multirow group 
                            # start with a leading & (empty first column)
                            # We replace that leading & with our part
                            lines[next_row_idx] = re.sub(r'^(\s*)&\s*', rf'\1{parts[p_idx]} & ', lines[next_row_idx])
                    
                    i += 1
                    continue
            # Center the multicolumn entries
            if '\\multicolumn' in line:
                line = re.sub(r'\\multicolumn\{(\d+)\}\{(l|r)\}', r'\\multicolumn{\1}{c}', line)

            # ad centering before the tabular environment
            if '\\begin{tabular}' in line:
                line = line.replace('\\begin{tabular}', '\\centering\n\\begin{tabular}')
            
            final_lines.append(line)
            i += 1
            
        return '\n'.join(final_lines)

    latex_code = distribute_multirow_lines(latex_code)
    # -----------------------------------------

    # 3. Final structural fixes
    if longtable:
        latex_code = latex_code.replace("\\begin{tabular}", "\\begin{longtable*}")
        latex_code = latex_code.replace("\\end{tabular}", "\\end{longtable*}")
        # and omit table environment
        latex_code = latex_code.replace("\\begin{table}", "")
        latex_code = latex_code.replace("\\end{table}", "")
    else:
        latex_code = latex_code.replace("\\begin{table}", "\\begin{table*}")
        latex_code = latex_code.replace("\\end{table}", "\\end{table*}")
    
    # Adjust header: Remove the "Task" label that Pandas puts in the MultiIndex header row
    latex_code = latex_code.replace("& Task &", "& &", 1)

    latex_code = latex_code.replace("NaN", "--")
    
    print(latex_code)

In [17]:
# Flagships
selected_models = [
    "daint_prod_ift_mask_fix_1N4n_9d30cad5",
    "daint_prod_ift_q3-4b_1N4n_16cdce0f",
]

In [18]:
# # Highlight
# latex_table_name = "tab:highlight-evals"
# selected_tasks = [
#     "gsm8k_cot_singleshot", 
#     "bbh_cot_fewshot", 
# ]
# summary_table = create_hierarchical_summary_table(
#     flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
#     sort_by="custom", 
#     include_step=False, 
#     # include_tps=True,
#     # include_tot_toks=True,
# )

# display(summary_table)
# dump_to_latex(summary_table)


In [19]:
# Main Math
latex_table_name = "tab:math-evals"
selected_tasks = [
    "gsm8k_cot_singleshot",
    "aime25", 
    "gpqa_main_cot_n_shot",
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=False,
    # include_tps=True,
    # include_tot_toks=True,
)
display(summary_table)
dump_to_latex(summary_table)

KeyError: 'summary_aime25/exact_match,none'

In [ ]:
# Main General
latex_table_name = "tab:general-evals"
selected_tasks = [
    "bbh_cot_fewshot", 
    "ifeval", 
    "cnn_dailymail_abisee"
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=False,
    # include_tps=True,
    # include_tot_toks=True,
)

display(summary_table)
dump_to_latex(summary_table)

In [ ]:
# Ablations L3 data
selected_models = [
    "daint_prod_ift_mask_fix_1N4n_9d30cad5",
    "daint_prod_ift_magpie_1N4n_44004b35",
]
latex_table_name = "tab:abl-l3-data-evals"
selected_tasks = [
    "gsm8k_cot_singleshot", 
    "bbh_cot_fewshot", 
    "ifeval", 
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=False,
    # include_tps=True,
    # include_tot_toks=True,
)

display(summary_table)
dump_to_latex(summary_table)


In [ ]:
# Ablations L3 supervision
# paper subset
selected_models = [
    "daint_prod_ift_mask_fix_1N4n_9d30cad5",
    "daint_prod_suprv_abl_1N4n_82938517",
    "daint_prod_suprv_abl_1N4n_b258ae13",
]
latex_table_name = "tab:abl-l3-suprv-evals-pt1"
selected_tasks = [
    "gsm8k_cot_singleshot", 
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=False,
)

display(summary_table)
dump_to_latex(summary_table)

# paper subset
selected_models = [
    "daint_prod_suprv_abl_1N4n_d30c404e",
    "daint_prod_suprv_abl_1N4n_e12fd460",
    "daint_prod_suprv_abl_1N4n_fcdeefba",
]
latex_table_name = "tab:abl-l3-suprv-evals-pt2"
selected_tasks = [
    "gsm8k_cot_singleshot", 
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=False,
)

display(summary_table)
dump_to_latex(summary_table)


In [ ]:
# static k updated comparison
selected_models = [
    "daint_prod_ift_mask_fix_1N4n_9d30cad5",
    "daint_prod_suprv_abl_1N4n_e12fd460",
    "daint_prod_pre_arxiv_extra_1N4n_8aa66673",
]
latex_table_name = "tab:abl-l3-suprv-evals-pt3"
selected_tasks = [
    "gsm8k_cot_singleshot", 
]
summary_table = create_hierarchical_summary_table(
    flat_df, tasks=selected_tasks, model_configs=[m for m in MODEL_CONFIGS if m["id"] in selected_models],
    sort_by="custom", include_step=True,
)

display(summary_table)
dump_to_latex(summary_table)

# Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

PLOT_STRATEGY_ORDER = [
    "nan k=1",
    "nan k=2",
    "nan k=3",
    "nan k=4",
    "nan k=5",
    # "nan k=8",
    # "nan k=16",
    "['conf_adapt', 0.995] k=16",
    "['conf_adapt', 0.99] k=16",
    "['conf_adapt', 0.98] k=16",
    "['conf_adapt', 0.97] k=16",
    "['conf_adapt', 0.96] k=16",
    "['conf_adapt', 0.95] k=16",
    "['conf_adapt', 0.9] k=16",
    "['conf_adapt', 0.87] k=16",
    "['conf_adapt', 0.85] k=16",
    "['conf_adapt', 0.8] k=16",
    "['conf_adapt', 0.75] k=16",
    "['conf_adapt', 0.7] k=16",
    "['conf_adapt', 0.65] k=16",
    "['conf_adapt', 0.6] k=16",
]

# Core theme anchors
C_BLUE = "#6394ED"    # cornflowerblue
C_PURP = "#7763ED"    # cornflowersanalogouspurple
C_GOLD = "#EDBC63"    # cornflowerscomplement

PLOT_STRATEGY_COLOR_MAP = {
    # Static Series: Green -> Yellow -> Red (No changes)
    "nan k=1": "#76C893", 
    "nan k=2": "#B5D33D", 
    "nan k=3": "#EDBC63", 
    "nan k=4": "#E76F51", 
    "nan k=5": "#D62828", 
    
    # ConfAdapt Series: Extended Blue -> Purple -> Indigo spectrum
    # High-confidence additions (very light cool tones)
    "['conf_adapt', 0.995] k=16": "#D0F0FD", # Ice Blue
    "['conf_adapt', 0.99] k=16":  "#BEE9FA", # Pale Sky
    "['conf_adapt', 0.98] k=16":  "#ADE2F7", # Light Sky
    "['conf_adapt', 0.97] k=16":  "#A0D9EF", # Previous Pale Sky Blue
    "['conf_adapt', 0.96] k=16":  "#91CEF2", # Soft Blue
    
    # Existing range (anchored to your paper colors)
    "['conf_adapt', 0.95] k=16":  "#82C1F5", 
    "['conf_adapt', 0.9] k=16":   "#6394ED", # Your Cornflower Blue (Anchor)
    "['conf_adapt', 0.87] k=16":  "#6B7CEE", 
    "['conf_adapt', 0.85] k=16":  "#7763ED", # Your Analogous Purple (Anchor)
    "['conf_adapt', 0.8] k=16":   "#6A4CD1", 
    "['conf_adapt', 0.75] k=16":  "#5E35B1", 
    "['conf_adapt', 0.7] k=16":   "#512DA8", 
    "['conf_adapt', 0.65] k=16":  "#4527A0", 
    "['conf_adapt', 0.6] k=16":   "#311B92", # Deepest Midnight Purple
}

PLOT_STRATEGY_MARKER_MAP = {
    # Static is Squares
    **{f"nan k={i}": "s" for i in range(1, 6)},
    # ConfAdapt is Circles
    **{k: "o" for k in PLOT_STRATEGY_COLOR_MAP.keys() if "conf_adapt" in k}
}

PLOT_STRATEGY_MAP = {
    "nan": "Static Baseline",
    **{f"['conf_adapt', {t}]": f"ConfAdapt ($\\tau={t}$)" 
        for t in ["0.995", "0.99", "0.98", "0.97", "0.96", "0.95", "0.9", "0.87", "0.85", "0.8", "0.75", "0.7", "0.65", "0.6"]}
}

def create_plots(
    df, 
    task_key, 
    selected_model_id, 
    sort_by="custom", 
    show_eff_vs_tps=False,
    figsize=(15, 8), 
    save_fig=False, 
    filename_prefix="dynamics",
    eff_axis_range = range(0,16+1,2),
    scheme_filter=lambda x: True,
    save_suffix="",
):
    """
    Generates three plots for a SINGLE model.
    Legend: 2 rows, centered under the first two plots.
    Colorbar: Horizontal, centered under the third plot.
    """
    strategy_key = "config_config_gen_kwargs_strategy"
    k_key = "config_config_gen_kwargs_k_toks"
    step_col = "summary__step"

    # Get model configuration and pretty name
    m_cfg = next((m for m in MODEL_CONFIGS if m["id"] == selected_model_id), None)
    if not m_cfg:
        print(f"Model ID {selected_model_id} not found in MODEL_CONFIGS.")
        return
    
    model_pretty_name = m_cfg.get("pretty_name", selected_model_id)
    t_cfg = TASK_CONFIGS[task_key]
    perf_col = t_cfg["perf_col"]
    
    # Metric naming and scaling
    perf_style_key = t_cfg.get("perf_style", "acc_pct")
    perf_meta = TASK_PERF_MAP.get(perf_style_key, {"pretty_name": perf_style_key})
    perf_pretty_label = perf_meta["pretty_name"]
    scale = 100 if "%" in perf_pretty_label else 1
    
    # Filter and prepare data
    subset = df[(df["name"] == selected_model_id) & (~df[perf_col].isnull())].copy()
    subset = apply_custom_filters(subset, m_cfg.get("filters", {}))
    
    if subset.empty:
        print(f"No data found for model {selected_model_id} on task {task_key}")
        return
        
    subset[strategy_key] = subset[strategy_key].astype(str)
    subset[k_key] = subset[k_key].fillna(1).astype(int)

    # Gather and sort curves
    all_curves = []
    for (strat, k_val), group in subset.groupby([strategy_key, k_key]):
        strat_logic_id = f"{strat} k={k_val}"
        if sort_by == "custom" and strat_logic_id not in PLOT_STRATEGY_ORDER:
            # print(f"Skipping strategy {strat_logic_id} as it's not in the custom order list.")
            continue

        if scheme_filter and not scheme_filter(strat_logic_id):
            print(f"Skipping strategy {strat_logic_id} due to scheme filter.")
            continue
        
        group = group.sort_values(step_col)
        final_row = group.iloc[-1]
        
        task_eff_col = t_cfg["eff_col"]
        # if any(key in selected_model_id for key in ["9d30cad5", "16cdce0f", "44004b35"]):
        if final_row[task_eff_col] is pd.NA or final_row[task_eff_col] == np.nan or str(final_row[task_eff_col]) == "nan":
            task_eff_col = task_eff_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")

        task_tps_col = t_cfg["tps_col"]
        # if any(key in selected_model_id for key in ["9d30cad5", "16cdce0f", "44004b35"]):
        if final_row[task_tps_col] is pd.NA or final_row[task_tps_col] == np.nan or str(final_row[task_tps_col]) == "nan":
            task_tps_col = task_tps_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")
        
        all_curves.append({
            "label": f"{PLOT_STRATEGY_MAP.get(strat, strat)} {f'k={k_val}' if strat == 'nan' else ''}",
            "logic_id": strat_logic_id,
            "steps": group[step_col].values,
            "perf": group[perf_col].values * scale,
            "eff": group[task_eff_col].values,
            "final_perf": final_row[perf_col] * scale,
            "final_eff": final_row[task_eff_col],
            "final_tps": final_row[task_tps_col],
        })

    if sort_by == "custom":
        all_curves.sort(key=lambda x: PLOT_STRATEGY_ORDER.index(x["logic_id"]))
    elif sort_by == "perf":
        all_curves.sort(key=lambda x: x["final_perf"], reverse=True)
    elif sort_by == "eff":
        all_curves.sort(key=lambda x: x["final_eff"])

    for c in all_curves:
        c.update({"label":tex_safe(c["label"])})

    # Initialize Figure
    if show_eff_vs_tps:
        fig, axes = plt.subplots(1, 4, figsize=figsize)
    else:
        fig, axes = plt.subplots(1, 3, figsize=figsize)
    # fig.suptitle(tex_safe(f"{model_pretty_name} | {t_cfg['pretty_name']}"), fontsize=16, y=0.98)
    
    # Plotting logic
    for c in all_curves:
        marker = PLOT_STRATEGY_MARKER_MAP.get(c["logic_id"], "o")
        color = PLOT_STRATEGY_COLOR_MAP.get(c["logic_id"], "black")
        assert marker is not None, f"Marker not found for strategy {c['logic_id']}"
        axes[0].plot(c["steps"], c["perf"], c=color, marker=marker, markersize=7, label=c["label"], alpha=0.7,markeredgecolor='black')
        axes[1].plot(c["steps"], c["eff"], c=color, marker=marker, markersize=7, alpha=0.7,markeredgecolor='black')
        
        # sc = axes[2].scatter(c["eff"], c["perf"], c=c["steps"], cmap='viridis', edgecolors='none', s=40, alpha=0.7)
        # axes[2].plot(c["eff"], c["perf"], marker=marker, alpha=0.3, linewidth=1.5)
        # only scatter the last point to reduce clutter and no lines
        # print(f"Plotting {c['logic_id']} with final perf {c['final_perf']:.2f} (type = {c['final_perf']}) and final eff {c['final_eff']:.2f} (type = {c['final_eff']})")
        sc = axes[2].scatter(c["eff"][-1], c["perf"][-1], c=color, marker=marker, edgecolors='black', s=50, alpha=0.7)

        if show_eff_vs_tps:
            axes[3].scatter(c["final_eff"], c["final_tps"], c=color, marker=marker, edgecolors='black', s=50, alpha=0.7)
    
    # add a 1-1 "perfect strong scaling" line by finding the setting for k=1 and scaling it by the k values
    if show_eff_vs_tps:
        c = next((curve for curve in all_curves if curve["logic_id"] == "nan k=1"), None)
        c_for_3x = next((curve for curve in all_curves if curve["logic_id"] == "nan k=3"), None)
        if c is None:
            print("No k=1 curve found for perfect strong scaling line.")
            return
        base_eff = c["final_eff"]
        base_tps = c["final_tps"]
        effs = np.array([curve["final_eff"] for curve in all_curves])
        effs = np.sort(effs)
        perfect_tps = base_tps * (effs / base_eff)
        axes[3].plot(effs, perfect_tps, linestyle='--', color='gray', alpha=0.5, label='Perfect Scaling')
        # # also calculate a more conservative estimate based on 1x-3x apparent slope extrapolated
        # _3x_eff = c_for_3x["final_eff"] if c_for_3x is not None else base_eff * 3
        # slope = (c_for_3x["final_tps"] - base_tps) / (_3x_eff - base_eff) if c_for_3x is not None else base_tps / base_eff
        # conservative_tps = base_tps + slope * (effs - base_eff)
        # axes[3].plot(effs, conservative_tps, linestyle=':', color='red', alpha=0.5, label='3x Rate Extrapol.')


    # Subplot Styling
    axes[0].set_ylabel(tex_safe(perf_pretty_label))
    axes[1].set_ylabel("Acceleration Factor")
    axes[2].set_ylabel(tex_safe(perf_pretty_label))
    axes[2].set_xlabel("Acceleration Factor")
    if show_eff_vs_tps:
        axes[3].set_xlabel("Acceleration Factor")
        axes[3].set_ylabel("Throughput (Toks/s)")

    if perf_style_key == "acc_pct":
        # set the axes[0] y axix ticks based max performance values rounded up to nearest 10
        max_perf = max(max(c["perf"]) for c in all_curves)
        y_tick_max = int(np.ceil(max_perf / 10.0)) * 10
        print(f"Max performance: {max_perf}, setting y_tick_max to {y_tick_max}")
        axes[0].set_yticks(list(range(0, y_tick_max, 10)) + [y_tick_max])
        axes[0].get_yaxis().set_major_formatter(plt.ScalarFormatter())
    else:
        # set 5 linear y ticks based on data range
        max_perf = max(max(c["perf"]) for c in all_curves)
        min_perf = min(min(c["perf"]) for c in all_curves)
        y_tick_step = (max_perf - min_perf) / 5
        y_ticks = [min_perf + i * y_tick_step for i in range(6)]
        axes[0].set_yticks(y_ticks)
        axes[0].get_yaxis().set_major_formatter(plt.ScalarFormatter())
    
    # set ticks for the acceleration factor axes
    axes[1].set_yticks(eff_axis_range)
    axes[1].get_yaxis().set_major_formatter(plt.ScalarFormatter())
    axes[2].set_xticks(eff_axis_range)
    axes[2].get_xaxis().set_major_formatter(plt.ScalarFormatter())
    if show_eff_vs_tps:
        axes[3].set_xticks(eff_axis_range)
        axes[3].get_xaxis().set_major_formatter(plt.ScalarFormatter())

    # make axes[2] yaxis match axes[0] limits and ticks
    axes[2].set_ylim(axes[0].get_ylim())
    axes[2].set_yticks(axes[0].get_yticks())
    axes[2].get_yaxis().set_major_formatter(plt.ScalarFormatter())
    
    for i, ax in enumerate(axes):
        if i < 2: ax.set_xlabel("Training Step")
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.subplots_adjust(bottom=0.28, top=0.88, wspace=0.3)

    # # Legend: 2 rows, centered under first two plots
    # num_cols = (len(all_curves) + 1) // 2
    # # num_cols = len(all_curves) + (1 if show_eff_vs_tps else 0)
    # fig.legend(
    #     loc='upper center', 
    #     # bbox_to_anchor=(0.37, 0.18), 
    #     bbox_to_anchor=(0.5, 0.18), 
    #     ncol=num_cols, 
    #     fontsize='small',
    #     frameon=False
    # )

    # # Two legends, one for the nan/static schemes and the other for the confadapt schemes
    # static_lines = []
    # confadapt_lines = []
    # static_labels = []
    # confadapt_labels = []
    # for line in axes[0].get_lines():
    #     label = line.get_label()
    #     if "ConfAdapt" not in label:
    #         static_lines.append(line)
    #         static_labels.append(label)
    #     else:
    #         confadapt_lines.append(line)
    #         confadapt_labels.append(label)
    # # Static legend
    # num_static_cols = (len(static_lines) + 1) // 2
    # fig.legend(
    #     static_lines, static_labels,
    #     loc='upper center', 
    #     bbox_to_anchor=(0.25, 0.18), 
    #     ncol=num_static_cols, 
    #     fontsize='small',
    #     frameon=False,
    #     title="Static Strategies",
    #     title_fontsize='medium',
    # )
    # # ConfAdapt legend
    # num_confadapt_cols = (len(confadapt_lines) + 1) // 2
    # fig.legend(
    #     confadapt_lines, confadapt_labels,
    #     loc='upper center', 
    #     bbox_to_anchor=(0.65, 0.18), 
    #     ncol=num_confadapt_cols, 
    #     fontsize='small',
    #     frameon=False, 
    #     title="ConfAdapt Strategies",
    #     title_fontsize='medium',
    # )

    import matplotlib.colors as mcolors
    import matplotlib.cm as cm  # Import the colormap module directly

    # Tune this to match your plot's alpha
    ALPHA = 0.7 

    # Helper to add alpha to hex colors
    def add_alpha(hex_list, alpha):
        return [mcolors.to_rgba(c, alpha=alpha) for c in hex_list]

    # 1. Setup Data for Colorbars
    cbar_plot_strat_color_map = {k: v for k, v in PLOT_STRATEGY_COLOR_MAP.items() if scheme_filter(k)}
    # static_colors = [cbar_plot_strat_color_map[f"nan k={i}"] for i in range(1, 6)]
    static_colors = [cbar_plot_strat_color_map[k] for k in PLOT_STRATEGY_ORDER if "nan" in k and scheme_filter(k)]
    static_colors = add_alpha(static_colors, ALPHA)
    conf_keys = [k for k in PLOT_STRATEGY_ORDER if "conf_adapt" in k and scheme_filter(k)]
    conf_colors = [PLOT_STRATEGY_COLOR_MAP[k] for k in conf_keys]
    
    conf_colors = add_alpha(conf_colors, ALPHA)
    conf_thresholds = [float(k.split(",")[1].split("]")[0].strip()) for k in conf_keys]

    # 2. Create the Colorbar Axes

    # 3. Static Colorbar (Squares)
    if len(static_colors) != 0:
        # Adjust these coordinates based on your specific figure size
        if len(conf_colors) != 0:
            ax_cb_static = fig.add_axes([0.15, 0.08, 0.3, 0.02]) 
        else:
            # center it
            ax_cb_static = fig.add_axes([0.35, 0.08, 0.3, 0.02])
        cmap_static = mcolors.ListedColormap(static_colors)
        norm_static = mcolors.BoundaryNorm(range(1, 7), cmap_static.N)
        # Use cm.ScalarMappable from the module, not the figure
        cb1 = fig.colorbar(
            cm.ScalarMappable(norm=norm_static, cmap=cmap_static),
            cax=ax_cb_static, orientation='horizontal',
            ticks=range(1, 6)
        )
        cb1.set_label(tex_safe('Static Strategy $k$'), fontsize='small', labelpad=5)
        # cb1.ax.set_xticklabels([f'{i}' for i in range(1, 6)], fontsize='x-small')
        # center the ticks on the color segments
        tick_locs = []
        boundaries = list(range(1, 7))
        for i in range(len(boundaries)-1):
            tick_locs.append((boundaries[i] + boundaries[i+1]) / 2)
        cb1.set_ticks(tick_locs)
        cb1.ax.set_xticklabels([f'{i}' for i in range(1, 6)], fontsize='x-small')

    # 4. ConfAdapt Colorbar (Circles)
    if len(conf_colors) != 0:
        # Adjust these coordinates based on your specific figure size
        if len(static_colors) != 0:
            ax_cb_conf   = fig.add_axes([0.55, 0.08, 0.3, 0.02])
        else:
            # center it
            ax_cb_conf   = fig.add_axes([0.35, 0.08, 0.3, 0.02])
        cmap_conf = mcolors.ListedColormap(conf_colors)
        # Reverse thresholds if they are descending to keep the bar left-to-right
        boundaries = [0.575] + [ (conf_thresholds[i] + conf_thresholds[i+1])/2 for i in range(len(conf_thresholds)-1) ] + [1.0]
        # Note: Boundaries must be increasing, so we flip them for the norm if needed
        boundaries = sorted(boundaries)
        norm_conf = mcolors.BoundaryNorm(boundaries, cmap_conf.N)

        cb2 = fig.colorbar(
            cm.ScalarMappable(norm=norm_conf, cmap=cmap_conf),
            cax=ax_cb_conf, orientation='horizontal'
        )
        cb2.set_label(tex_safe('ConfAdapt Threshold $\\tau$'), fontsize='small', labelpad=5)

        # Labeling every other threshold to keep it clean
        cb2.set_ticks(conf_thresholds)
        # tick_labels = [f'{t:.2f}' if i % 2 == 0 else '' for i, t in enumerate(conf_thresholds)]
        tick_labels = [f'{t:.2f}' if i % 1 == 0 else '' for i, t in enumerate(conf_thresholds)]
        # cb2.ax.set_xticklabels(tick_labels, fontsize='x-small')
        # center the ticks on the color segments
        tick_locs = []
        for i in range(len(boundaries)-1):
        # for i in range(len(boundaries)-1,0,-1):
            tick_locs.append((boundaries[i] + boundaries[i+1]) / 2)
        cb2.set_ticks(tick_locs)
        cb2.ax.set_xticklabels(tick_labels, fontsize='x-small')

    # Final layout adjustment
    fig.subplots_adjust(bottom=0.25)

    # # Colorbar: Horizontal, centered under the third plot (shifted left from previous version)
    # # [left, bottom, width, height] - decreased 'left' to 0.70 to center better
    # cbar_ax = fig.add_axes([0.69, 0.14, 0.22, 0.025]) 
    # cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
    # cbar.set_label('Training Step', fontsize=9)
    # cbar.ax.tick_params(labelsize=8)

    if save_fig:
        out_name = f"{filename_prefix}_{selected_model_id}_{task_key}{save_suffix}"
        full_path = os.path.join(BASE_FIGURE_DIR, out_name)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        plt.savefig(f"{full_path}.png", bbox_inches='tight')
        plt.savefig(f"{full_path}.pdf", bbox_inches='tight')
        print(f"Saved figure to {full_path}.png and .pdf")
    
    plt.show()

In [ ]:
def create_plots_scatter_only(
    df, 
    task_key, 
    selected_model_id, 
    sort_by="custom", 
    show_eff_vs_tps=False,
    figsize=(8, 14), 
    save_fig=False, 
    filename_prefix="dynamics",
    eff_axis_range = range(0,16+1,2),
    scheme_filter=lambda x: True,
    save_suffix="",
    render_step_plots=True,
    render_scatter_only=False,
    bottom_margin_override=None,
    cb_static_y_override=None,
    cb_conf_y_override=None,
    cbar_height_factor=0.015,
    # Flags for vertical colorbars
    vertical_cbar=False,
    right_margin_override=None,
    cb_static_x_override=None,
    cb_conf_x_override=None,
):
    """
    Generates plots vertically stacked. 
    Dynamically adjusts colorbar spacing if only the scatter plot is rendered.
    Optionally stacks colorbars vertically on the right.
    """
    strategy_key = "config_config_gen_kwargs_strategy"
    k_key = "config_config_gen_kwargs_k_toks"
    step_col = "summary__step"

    m_cfg = next((m for m in MODEL_CONFIGS if m["id"] == selected_model_id), None)
    if not m_cfg: return
    
    model_pretty_name = m_cfg.get("pretty_name", selected_model_id)
    t_cfg = TASK_CONFIGS[task_key]
    perf_col, perf_style_key = t_cfg["perf_col"], t_cfg.get("perf_style", "acc_pct")
    perf_meta = TASK_PERF_MAP.get(perf_style_key, {"pretty_name": perf_style_key})
    perf_pretty_label = perf_meta["pretty_name"]
    scale = 100 if "%" in perf_pretty_label else 1
    
    subset = df[(df["name"] == selected_model_id) & (~df[perf_col].isnull())].copy()
    subset = apply_custom_filters(subset, m_cfg.get("filters", {}))
    if subset.empty: return
        
    subset[strategy_key] = subset[strategy_key].astype(str)
    subset[k_key] = subset[k_key].fillna(1).astype(int)

    all_curves = []
    for (strat, k_val), group in subset.groupby([strategy_key, k_key]):
        strat_logic_id = f"{strat} k={k_val}"
        if (sort_by == "custom" and strat_logic_id not in PLOT_STRATEGY_ORDER) or (scheme_filter and not scheme_filter(strat_logic_id)):
            continue
        
        group = group.sort_values(step_col)
        final_row = group.iloc[-1]
        task_eff_col, task_tps_col = t_cfg["eff_col"], t_cfg["tps_col"]
        
        if any(str(final_row[c]) == "nan" for c in [task_eff_col, task_tps_col]):
            task_eff_col = task_eff_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")
            task_tps_col = task_tps_col.replace("summary_gsm8k_cot_singleshot/samples","summary_samples")

        all_curves.append({
            "label": f"{PLOT_STRATEGY_MAP.get(strat, strat)} {f'k={k_val}' if strat == 'nan' else ''}",
            "logic_id": strat_logic_id, "steps": group[step_col].values, "perf": group[perf_col].values * scale,
            "eff": group[task_eff_col].values, "final_perf": final_row[perf_col] * scale,
            "final_eff": final_row[task_eff_col], "final_tps": final_row[task_tps_col],
        })

    if sort_by == "custom": all_curves.sort(key=lambda x: PLOT_STRATEGY_ORDER.index(x["logic_id"]))
    for c in all_curves: c.update({"label":tex_safe(c["label"])})

    # Layout Logic
    if render_scatter_only:
        active_plots = ["scatter"]
        if show_eff_vs_tps: active_plots.append("tps")
        plot_height = 5 if not show_eff_vs_tps else 8
        figsize = (figsize[0], plot_height * (figsize[1]/14))
    elif render_step_plots:
        active_plots = ["perf_step", "eff_step", "scatter"]
        if show_eff_vs_tps: active_plots.append("tps")
    else:
        active_plots = ["scatter"]

    fig, axes_raw = plt.subplots(len(active_plots), 1, figsize=figsize)
    axes = axes_raw if len(active_plots) > 1 else [axes_raw]
    plot_to_ax = {p_type: axes[i] for i, p_type in enumerate(active_plots)}

    for c in all_curves:
        marker = PLOT_STRATEGY_MARKER_MAP.get(c["logic_id"], "o")
        color = PLOT_STRATEGY_COLOR_MAP.get(c["logic_id"], "black")
        if "perf_step" in plot_to_ax:
            plot_to_ax["perf_step"].plot(c["steps"], c["perf"], c=color, marker=marker, markersize=7, alpha=0.7, markeredgecolor='black')
        if "eff_step" in plot_to_ax:
            plot_to_ax["eff_step"].plot(c["steps"], c["eff"], c=color, marker=marker, markersize=7, alpha=0.7, markeredgecolor='black')
        if "scatter" in plot_to_ax:
            plot_to_ax["scatter"].scatter(c["eff"][-1], c["perf"][-1], c=color, marker=marker, edgecolors='black', s=50, alpha=0.7)
        if "tps" in plot_to_ax:
            plot_to_ax["tps"].scatter(c["final_eff"], c["final_tps"], c=color, marker=marker, edgecolors='black', s=50, alpha=0.7)

    # Perfect Scaling Line
    if "tps" in plot_to_ax:
        c_k1 = next((curve for curve in all_curves if curve["logic_id"] == "nan k=1"), None)
        if c_k1:
            base_eff, base_tps = c_k1["final_eff"], c_k1["final_tps"]
            effs = np.sort(np.array([curve["final_eff"] for curve in all_curves]))
            perfect_tps = base_tps * (effs / base_eff)
            plot_to_ax["tps"].plot(effs, perfect_tps, linestyle='--', color='gray', alpha=0.5, label='Perfect Scaling')

    for p_type, ax in plot_to_ax.items():
        if "step" in p_type: ax.set_xlabel("Training Step")
        else: ax.set_xlabel("Acceleration Factor")
        if "perf" in p_type or p_type == "scatter": ax.set_ylabel(tex_safe(perf_pretty_label))
        if p_type == "eff_step": ax.set_ylabel("Acceleration Factor")
        if p_type == "tps": ax.set_ylabel("Throughput (Toks/s)")
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    if "perf_step" in plot_to_ax and "scatter" in plot_to_ax:
        plot_to_ax["scatter"].set_ylim(plot_to_ax["perf_step"].get_ylim())

    # Dynamic Spacing for Colorbars
    h_scale = 14 / figsize[1]
    w_scale = 8 / figsize[0]
    
    if vertical_cbar:
        right_margin = (0.82) if right_margin_override is None else right_margin_override
        bottom_margin = 0.12 if bottom_margin_override is None else bottom_margin_override
        cb_static_x = (0.93) if cb_static_x_override is None else cb_static_x_override
        cb_conf_x = (0.86) if cb_conf_x_override is None else cb_conf_x_override
    else:
        if render_scatter_only:
            bottom_margin = (0.25 * h_scale) if bottom_margin_override is None else bottom_margin_override
            cb_static_y = (0.12 * h_scale) if cb_static_y_override is None else cb_static_y_override
            cb_conf_y = (0.05 * h_scale) if cb_conf_y_override is None else cb_conf_y_override
        else:
            bottom_margin = (0.18 * h_scale) if bottom_margin_override is None else bottom_margin_override
            cb_static_y = (0.08 * h_scale) if cb_static_y_override is None else cb_static_y_override
            cb_conf_y = (0.03 * h_scale) if cb_conf_y_override is None else cb_conf_y_override

    import matplotlib.colors as mcolors
    import matplotlib.cm as cm
    
    # Static data
    static_colors = [mcolors.to_rgba(PLOT_STRATEGY_COLOR_MAP[k], alpha=0.7) for k in PLOT_STRATEGY_ORDER if "nan" in k and scheme_filter(k)]
    static_labels = [k.split('=')[1] for k in PLOT_STRATEGY_ORDER if "nan" in k and scheme_filter(k)]
    
    # Conf data
    conf_keys = [k for k in PLOT_STRATEGY_ORDER if "conf_adapt" in k and scheme_filter(k)]
    conf_colors = [mcolors.to_rgba(PLOT_STRATEGY_COLOR_MAP[k], alpha=0.7) for k in conf_keys]
    conf_thresholds = [float(k.split(",")[1].split("]")[0].strip()) for k in conf_keys]

    cbar_thick = cbar_height_factor * (w_scale if vertical_cbar else h_scale)
    orientation = 'vertical' if vertical_cbar else 'horizontal'

    # Reversed Stacking: Conf on Top (0.55), Static on Bottom (0.15) if vertical
    if len(conf_colors) > 0:
        if vertical_cbar:
            ax_cb_conf = fig.add_axes([cb_conf_x, 0.45, cbar_thick, 0.45])
            # Reverse thresholds for top-to-bottom reading
            conf_colors_v = conf_colors[::-1]
            conf_thresholds_v = conf_thresholds[::-1]
        else:
            ax_cb_conf = fig.add_axes([0.35, cb_conf_y, 0.6, cbar_thick])
            conf_colors_v = conf_colors
            conf_thresholds_v = conf_thresholds
            
        cmap_conf = mcolors.ListedColormap(conf_colors_v)
        boundaries = sorted([0.575] + [(conf_thresholds[i] + conf_thresholds[i+1])/2 for i in range(len(conf_thresholds)-1)] + [1.0])
        norm_conf = mcolors.BoundaryNorm(boundaries, cmap_conf.N)
        cb2 = fig.colorbar(cm.ScalarMappable(norm=norm_conf, cmap=cmap_conf), cax=ax_cb_conf, orientation=orientation)
        cb2.set_label(tex_safe('ConfAdapt $\\tau$'), fontsize='small')
        
        tick_locs = [(boundaries[i] + boundaries[i+1]) / 2 for i in range(len(boundaries)-1)]
        cb2.set_ticks(tick_locs)
        # tick_labels = [f'{t:.2f}' for t in conf_thresholds_v]
        tick_labels = [f'{str(t)}' for t in conf_thresholds_v]
        
        if vertical_cbar:
            cb2.ax.set_yticklabels(tick_labels, fontsize='x-small')
        else:
            cb2.ax.set_xticklabels(tick_labels, fontsize='x-small')

    if len(static_colors) > 0:
        if vertical_cbar:
            ax_cb_static = fig.add_axes([cb_static_x, 0.15, cbar_thick, 0.25])
            # Reverse for top-to-bottom reading
            static_colors_v = static_colors[::-1]
            static_labels_v = static_labels[::-1]
        else:
            ax_cb_static = fig.add_axes([0.0, cb_static_y, 0.3, cbar_thick]) 
            static_colors_v = static_colors
            static_labels_v = static_labels
            
        cmap_static = mcolors.ListedColormap(static_colors_v)
        norm_static = mcolors.BoundaryNorm(range(1, len(static_colors_v) + 2), cmap_static.N)
        cb1 = fig.colorbar(cm.ScalarMappable(norm=norm_static, cmap=cmap_static), cax=ax_cb_static, orientation=orientation)
        cb1.set_label(tex_safe('Static $k$'), fontsize='small')
        
        tick_locs = np.arange(1.5, len(static_colors_v) + 1.5)
        cb1.set_ticks(tick_locs)
        
        if vertical_cbar:
            cb1.ax.set_yticklabels(static_labels_v, fontsize='x-small')
        else:
            cb1.ax.set_xticklabels(static_labels_v, fontsize='x-small')

    if vertical_cbar:
        plt.subplots_adjust(hspace=0.4, right=right_margin, bottom=bottom_margin, top=0.96)
    else:
        plt.subplots_adjust(hspace=0.4, bottom=bottom_margin, top=0.96)

    if save_fig:
        out_name = f"{filename_prefix}_{selected_model_id}_{task_key}{save_suffix}"
        full_path = os.path.join(BASE_FIGURE_DIR, out_name)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        plt.savefig(f"{full_path}.png", bbox_inches='tight')
        plt.savefig(f"{full_path}.pdf", bbox_inches='tight')
    
    plt.show()

In [ ]:
SAVE_FIGS = False
# SAVE_FIGS = True

In [ ]:
# Flagship L3 on GSM
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,7+1,1),
)
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,7+1,1),
    scheme_filter=lambda x: "conf_adapt" in x,
    save_suffix="_ca_only",
)
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
    scheme_filter=lambda x: "nan" in x, 
    save_suffix="_static_only",
)
create_plots_scatter_only(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(4,8),
    render_step_plots=False,
    render_scatter_only=True,
    save_suffix="_scatter_only",
    cbar_height_factor=0.02,
    vertical_cbar=True,
    cb_static_x_override=0.85,
    cb_conf_x_override=0.85,
    # vertical_cbar=False,
    # bottom_margin_override=0.3,
    # cb_static_y_override=0.0,
    # cb_conf_y_override=0.0,
)

In [ ]:
# Flagship Q3 on GSM
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
)
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,5+1,1),
    scheme_filter=lambda x: "conf_adapt" in x,
    save_suffix="_ca_only",
)
create_plots(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
    scheme_filter=lambda x: "nan" in x, 
    save_suffix="_static_only",
)
create_plots_scatter_only(
    flat_df, 
    task_key="gsm8k_cot_singleshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(4,8),
    render_step_plots=False,
    render_scatter_only=True,
    save_suffix="_scatter_only",
    cbar_height_factor=0.02,
    vertical_cbar=True,
    cb_static_x_override=0.85,
    cb_conf_x_override=0.85,
    # vertical_cbar=False,
    # bottom_margin_override=0.3,
    # cb_static_y_override=0.0,
    # cb_conf_y_override=0.0,
)

In [ ]:
# # Flagship L3 on BBH
# create_plots(
#     flat_df, 
#     task_key="bbh_cot_fewshot",
#     selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
#     figsize=(14, 4),
#     save_fig=SAVE_FIGS,
#     # show_eff_vs_tps=True,
#     # figsize=(18, 4),
#     eff_axis_range=range(0,6+1,1)
# )

# Flagship L3 on BBH
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,7+1,1),
)
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,7+1,1),
    scheme_filter=lambda x: "conf_adapt" in x,
    save_suffix="_ca_only",
)
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
    scheme_filter=lambda x: "nan" in x, 
    save_suffix="_static_only",
)
create_plots_scatter_only(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_mask_fix_1N4n_9d30cad5", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(4,8),
    render_step_plots=False,
    render_scatter_only=True,
    save_suffix="_scatter_only",
    cbar_height_factor=0.02,
    vertical_cbar=True,
    cb_static_x_override=0.85,
    cb_conf_x_override=0.85,
    # vertical_cbar=False,
    # bottom_margin_override=0.3,
    # cb_static_y_override=0.0,
    # cb_conf_y_override=0.0,
)

In [ ]:
# # Flagship Q3 on BBH
# create_plots(
#     flat_df, 
#     task_key="bbh_cot_fewshot", 
#     selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
#     figsize=(14, 4),
#     save_fig=SAVE_FIGS,
#     # show_eff_vs_tps=True,
#     # figsize=(18, 4),
#     eff_axis_range=range(0,6+1,1)
# )

# Flagship Q3 on BBH
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
)
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,5+1,1),
    scheme_filter=lambda x: "conf_adapt" in x,
    save_suffix="_ca_only",
)
create_plots(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(14, 4),
    eff_axis_range=range(0,6+1,1),
    scheme_filter=lambda x: "nan" in x, 
    save_suffix="_static_only",
)
create_plots_scatter_only(
    flat_df, 
    task_key="bbh_cot_fewshot", 
    selected_model_id="daint_prod_ift_q3-4b_1N4n_16cdce0f", 
    save_fig=SAVE_FIGS,
    show_eff_vs_tps=False,
    figsize=(4,8),
    render_step_plots=False,
    render_scatter_only=True,
    save_suffix="_scatter_only",
    cbar_height_factor=0.02,
    vertical_cbar=True,
    cb_static_x_override=0.85,
    cb_conf_x_override=0.85,
    # vertical_cbar=False,
    # bottom_margin_override=0.3,
    # cb_static_y_override=0.0,
    # cb_conf_y_override=0.0,
)